In [ ]:
# bowaka_v2_lab notebook bootstrap cell — DO NOT EDIT BY HAND.
# Adds the lab's src/ (and its bowaka_common dependency) to sys.path and pins
# the working directory to the repo root, so `import bowaka_v2_lab` and
# repo-root-relative CONFIG_PATH parameters resolve identically under jupyter,
# papermill, and the QuantsLab scheduler.
import os
import sys
from pathlib import Path

_lab_root = None
for _candidate in [Path.cwd(), *Path.cwd().parents]:
    if (_candidate / "src" / "bowaka_v2_lab" / "__init__.py").is_file():
        _lab_root = _candidate
        break
if _lab_root is None:
    raise RuntimeError(
        f"bowaka_v2_lab bootstrap: src/bowaka_v2_lab/ not found at or above {Path.cwd()}"
    )

# Pin CWD to the repo root so repo-root-relative CONFIG_PATH values resolve
# regardless of how the notebook was launched (jupyter CWD = notebook dir,
# scheduler = repo root). The lab ALWAYS lives at
# ``<repo_root>/research_notebooks/bowaka_v2_lab``, so derive the repo root from
# that canonical layout. A marker-only heuristic ("a dir with research_notebooks/
# AND Makefile") mis-matches the lab dir itself when it carries a Makefile and a
# stray nested research_notebooks/ — which then chdir's one level too deep and
# breaks every repo-root-relative path.
if _lab_root.parent.name == "research_notebooks":
    _repo_root = _lab_root.parent.parent
else:
    # Fallback: the repo root holds research_notebooks/bowaka_common (the sibling
    # package) — a marker a stray nested research_notebooks/ inside the lab lacks.
    _repo_root = _lab_root
    for _candidate in [_lab_root, *_lab_root.parents]:
        if (_candidate / "research_notebooks" / "bowaka_common").is_dir():
            _repo_root = _candidate
            break
os.chdir(_repo_root)

# Make the lab and its bowaka_common dependency importable from the working
# tree, even when the packages are not pip-installed. v1 bowaka_lab is
# deliberately excluded — v2 must not import v1.
for _src in (_lab_root / "src",
             _repo_root / "research_notebooks" / "bowaka_common" / "src"):
    if _src.is_dir() and str(_src) not in sys.path:
        sys.path.insert(0, str(_src))

import bowaka_v2_lab  # noqa: F401
print(f"bowaka_v2_lab {bowaka_v2_lab.__version__} (cwd={_repo_root})")


In [ ]:
# Papermill parameter cell.
CONFIG_PATH = 'research_notebooks/bowaka_v2_lab/configs/bowaka_v2_backtest_smoke.yml'


# 06 — Execution Cost & Liquidity Study

Runs a real backtest, then profiles its entry decisions by ADV bucket, spread bucket, and time of day.

In [ ]:
import pandas as pd
from bowaka_v2_lab.backtest_runner import run_config_backtest
from bowaka_v2_lab.reports.liquidity_execution import (adv_bucket_distribution,
  spread_bucket_distribution, time_of_day_buckets)
result = run_config_backtest(CONFIG_PATH)
print('backtest summary:', result.summary)
dec_path = result.run_dir / 'entry_decisions.parquet'
if dec_path.is_file():
    decisions = pd.read_parquet(dec_path)
    print('entry decisions:', len(decisions))
    print('--- ADV-bucket distribution ---')
    print(adv_bucket_distribution(decisions))
    print('--- spread-bucket distribution ---')
    print(spread_bucket_distribution(decisions))
    print('--- time-of-day distribution ---')
    print(time_of_day_buckets(decisions))
else:
    print('no entry_decisions.parquet at', dec_path)
